# Migrations

A migration applies a versioned schema change.

In [ ]:
# A migration has a unique revision and reversible schema operations.
migration = {
    "revision": "0002",
    "upgrade": "ALTER TABLE tasks ADD COLUMN completed BOOLEAN DEFAULT FALSE;",
    "downgrade": "ALTER TABLE tasks DROP COLUMN completed;",
}

for key, value in migration.items():
    print(f"{key}: {value}")

Review and test both upgrade and rollback paths before production.

## Polished version

A standalone migration has an identifier plus executable upgrade and downgrade functions, mirroring Alembic's reversible workflow.

In [ ]:
import sqlite3
from collections.abc import Callable
from dataclasses import dataclass


# Store upgrade and downgrade functions together as one versioned change.
@dataclass(frozen=True)
class Migration:
    revision: str
    upgrade: Callable[[sqlite3.Connection], None]
    downgrade: Callable[[sqlite3.Connection], None]


# Keep each schema operation small so it can be tested independently.
def add_completed(connection: sqlite3.Connection) -> None:
    connection.execute(
        "ALTER TABLE tasks ADD COLUMN completed INTEGER NOT NULL DEFAULT 0"
    )


def remove_completed(connection: sqlite3.Connection) -> None:
    connection.execute("ALTER TABLE tasks DROP COLUMN completed")


migration = Migration(
    revision="0002_add_completed",
    upgrade=add_completed,
    downgrade=remove_completed,
)

connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL)")

# The context manager commits on success and rolls back on failure.
with connection:
    migration.upgrade(connection)
print("After upgrade:", [row[1] for row in connection.execute("PRAGMA table_info(tasks)")])

# Downgrade reverses the same revision when rollback is required.
with connection:
    migration.downgrade(connection)
print("After downgrade:", [row[1] for row in connection.execute("PRAGMA table_info(tasks)")])

## Applied in this repository

The REST project uses the same revision, upgrade, and downgrade shape in [its Alembic migration](../00P1-project-rest-api/migrations/versions/0001_create_users_and_tasks.py).